# Assignment 06 · Notebook 06: đối chuẩn PyTorch và Keras, 16 mô hình

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Mọi chỉ số tính lại từ dự báo đã lưu (`outputs/preds/`) bằng cùng một hàm cho cả hai framework. Độ trễ suy luận đo
trên **CPU, 1 luồng** cho cả hai framework: khởi động 20 lần, lấy trung vị 200 lần với lô 1, và trung vị 30 lần với
lô 256 để tính thông lượng. Mô hình Keras chạy qua `tf.function` (đồ thị đã biên dịch), mô hình PyTorch
chạy eager như cách dùng thông thường. Notebook cũng gộp `models/metadata.json` cho Web App.

In [1]:
import sys
sys.path.insert(0, "..")
from a06.models_keras import configure_tf
configure_tf(1)                       # phải đặt trước phép tính TF đầu tiên
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
import torch
from a06.bench import (eval_churn, eval_stock, latency, load_preds, merge_metadata, naive_stock,
                       sigmoid)
from a06.data_churn import FEATURES as KK_FEATURES, load_churn
from a06.data_stock import prepare_stock
from a06.export import MODELS, save_metrics, write_dat
from a06.models_torch import CELLS, CELL_LABELS, build_torch
from a06.train_torch import CFG, OPT

torch.set_num_threads(1)
S, C = prepare_stock(), load_churn()
DATA = {"amzn": S, "kkbox": C}
FW = {"torch": "PyTorch", "keras": "Keras"}
import json
meta_runs = {}
for p in sorted(MODELS.glob("meta_*.json")):
    meta_runs.update(json.loads(p.read_text(encoding="utf-8")))
print(len(meta_runs), "mô hình đã huấn luyện")

16 mô hình đã huấn luyện


## Chỉ số trên tập test, tính lại cho cả 16 mô hình

In [2]:
res = {}
for ds, D in DATA.items():
    for fw in FW:
        for cell in CELLS:
            key = f"{ds}_{fw}_{cell}"
            pv, pt = load_preds(key)
            r = eval_stock(D, pt) if ds == "amzn" else eval_churn(D, pv, pt)
            r.pop("price_pred", None)
            res[key] = {**r, **{k: meta_runs[key][k] for k in ("params", "best_epoch", "epochs_run", "epoch_s",
                                                              "train_s", "device", "best_val_loss", "file")}}
naive = naive_stock(S)

## Độ trễ suy luận trên CPU (1 luồng)

In [3]:
def load(ds, fw, cell):
    path = MODELS / meta_runs[f"{ds}_{fw}_{cell}"]["file"]
    if fw == "torch":
        m = build_torch(cell)
        m.load_state_dict(torch.load(path, map_location="cpu"))
        m.eval()
        def run(X):
            with torch.no_grad():
                return m(torch.as_tensor(X))
        return run
    m = keras.models.load_model(path)
    f = tf.function(lambda X: m(X, training=False), reduce_retracing=True)   # đồ thị đã biên dịch, như khi phục vụ
    return lambda X: f(X)

# Đo hết PyTorch rồi mới tới Keras: luồng của TensorFlow còn quay chờ sau mỗi lần gọi, nếu xen kẽ sẽ
# giành mất lõi CPU duy nhất của PyTorch và làm độ trễ PyTorch đo được tăng vài lần.
for fw in FW:
    for ds, D in DATA.items():
        sample = D["X"]["test"][:256]
        for cell in CELLS:
            key = f"{ds}_{fw}_{cell}"
            res[key].update(latency(load(ds, fw, cell), sample))
            res[key]["size_kb"] = (MODELS / res[key]["file"]).stat().st_size / 1024
            print(f"{key:22s} {res[key]['latency_ms']:7.3f} ms/mẫu · {res[key]['throughput']:9.0f} mẫu/s · {res[key]['size_kb']:6.1f} KB")

amzn_torch_rnn           0.354 ms/mẫu ·     71730 mẫu/s ·   29.8 KB


amzn_torch_lstm          0.270 ms/mẫu ·     45763 mẫu/s ·   85.4 KB


amzn_torch_gru           0.787 ms/mẫu ·     34920 mẫu/s ·   66.8 KB


amzn_torch_bilstm        0.438 ms/mẫu ·     24150 mẫu/s ·  167.8 KB
kkbox_torch_rnn          0.376 ms/mẫu ·     72651 mẫu/s ·   29.9 KB


kkbox_torch_lstm         0.264 ms/mẫu ·     46941 mẫu/s ·   85.4 KB


...

amzn_keras_bilstm        2.756 ms/mẫu ·     11394 mẫu/s ·  525.8 KB


kkbox_keras_rnn          0.923 ms/mẫu ·     99238 mẫu/s ·  108.2 KB


kkbox_keras_lstm         1.143 ms/mẫu ·     24985 mẫu/s ·  272.5 KB


kkbox_keras_gru          1.562 ms/mẫu ·     32314 mẫu/s ·  220.0 KB


kkbox_keras_bilstm       2.079 ms/mẫu ·     10989 mẫu/s ·  525.9 KB


## Bảng đối chuẩn

In [4]:
cols_s = ["rmse", "mae", "r2", "r2_ret", "dir_acc", "params", "size_kb", "epoch_s", "latency_ms", "throughput"]
tab_s = pd.DataFrame({f"{FW[fw]} {CELL_LABELS[c]}": {k: res[f"amzn_{fw}_{c}"][k] for k in cols_s}
                      for fw in FW for c in CELLS}).T
tab_s.loc["Mốc ngây thơ", ["rmse", "mae", "r2", "r2_ret"]] = [naive[k] for k in ("rmse", "mae", "r2", "r2_ret")]
tab_s.round(4)

,rmse,mae,r2,r2_ret,dir_acc,params,size_kb,epoch_s,latency_ms,throughput
PyTorch Simple RNN,3.1371,2.2902,0.9880,-0.0148,0.4843,6849.0,29.8496,0.2506,0.3541,71729.7805
PyTorch LSTM,3.1309,2.2755,0.9880,-0.0063,0.5147,21057.0,85.3604,0.2149,0.2698,45762.9088
PyTorch GRU,3.1409,2.2858,0.9879,-0.0136,0.5059,16321.0,66.8496,0.1694,0.7870,34919.7256
PyTorch BiLSTM,3.1288,2.2776,0.9880,-0.0019,0.5088,42049.0,167.7568,0.2714,0.4382,24150.4877
Keras Simple RNN,3.2174,2.3575,0.9874,-0.0477,0.4980,6785.0,108.1377,0.9260,0.9219,103080.3302
Keras LSTM,3.1321,2.2817,0.9880,-0.0037,0.5117,20801.0,272.4570,1.4479,1.2477,24318.4193
Keras GRU,3.1321,2.2754,0.9880,-0.0041,0.5215,16321.0,219.9492,1.4503,1.3913,30396.9413
Keras BiLSTM,3.1401,2.2835,0.9880,-0.0136,0.5166,41537.0,525.8447,2.0690,2.7562,11394.4643
Mốc ngây thơ,3.1239,2.2741,0.9881,-0.0000,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
cols_c = ["acc", "precision", "recall", "f1", "roc_auc", "pr_auc", "params", "size_kb", "epoch_s", "latency_ms", "throughput"]
pd.DataFrame({f"{FW[fw]} {CELL_LABELS[c]}": {k: res[f"kkbox_{fw}_{c}"][k] for k in cols_c}
              for fw in FW for c in CELLS}).T.round(4)

,acc,precision,recall,f1,roc_auc,pr_auc,params,size_kb,epoch_s,latency_ms,throughput
PyTorch Simple RNN,0.8865,0.3483,0.3058,0.3257,0.6703,0.2835,6849.0,29.8604,0.5913,0.3762,72650.9067
PyTorch LSTM,0.8933,0.3826,0.3103,0.3426,0.6819,0.3102,21057.0,85.3711,0.5972,0.2644,46941.4699
PyTorch GRU,0.9036,0.4391,0.2738,0.3373,0.6802,0.3004,16321.0,66.8604,0.7517,0.8927,30302.1336
PyTorch BiLSTM,0.8907,0.3714,0.3170,0.3420,0.6801,0.2975,42049.0,167.7676,2.3099,0.3955,25556.4263
Keras Simple RNN,0.8991,0.4051,0.2701,0.3241,0.6616,0.2697,6785.0,108.2041,2.7434,0.9225,99238.2688
Keras LSTM,0.9022,0.4294,0.2783,0.3377,0.6787,0.3046,20801.0,272.5195,16.8719,1.1430,24985.4821
Keras GRU,0.8976,0.4026,0.2954,0.3408,0.6797,0.3091,16321.0,220.0117,19.3455,1.5623,32313.6443
Keras BiLSTM,0.8983,0.4068,0.2939,0.3413,0.6776,0.3028,41537.0,525.9072,53.4203,2.0791,10988.6572


## Tổng hợp so sánh hai framework

Chênh lệch chỉ số giữa hai framework trên cùng kiến trúc, và tỉ số độ trễ Keras / PyTorch.

In [6]:
cmp = {}
for ds, metric in (("amzn", "rmse"), ("kkbox", "f1")):
    for c in CELLS:
        t, k = res[f"{ds}_torch_{c}"], res[f"{ds}_keras_{c}"]
        cmp[f"{ds}_{c}"] = {"diff": k[metric] - t[metric], "lat_ratio": k["latency_ms"] / t["latency_ms"],
                            "thr_ratio": t["throughput"] / k["throughput"], "param_diff": k["params"] - t["params"]}
lat = {fw: np.median([res[f"{ds}_{fw}_{c}"]["latency_ms"] for ds in DATA for c in CELLS]) for fw in FW}
best = {f"{ds}_{fw}": min(CELLS, key=lambda c: res[f"{ds}_{fw}_{c}"]["best_val_loss"]) for ds in DATA for fw in FW}
best_test = {"amzn": min(res, key=lambda k: res[k]["rmse"] if k.startswith("amzn") else np.inf),
             "kkbox": max(res, key=lambda k: res[k]["f1"] if k.startswith("kkbox") else -np.inf)}
beats_naive = [k for k in res if k.startswith("amzn") and res[k]["rmse"] < naive["rmse"]]
print("mô hình tốt nhất theo val:", best)
print("tốt nhất trên test:", best_test, "· số mô hình AMZN thắng mốc ngây thơ:", len(beats_naive), beats_naive)
print("độ trễ trung vị (ms):", lat)
pd.DataFrame(cmp).T.round(4)

mô hình tốt nhất theo val: {'amzn_torch': 'gru', 'amzn_keras': 'lstm', 'kkbox_torch': 'lstm', 'kkbox_keras': 'gru'}
tốt nhất trên test: {'amzn': 'amzn_torch_bilstm', 'kkbox': 'kkbox_torch_lstm'} · số mô hình AMZN thắng mốc ngây thơ: 0 []
độ trễ trung vị (ms): {'torch': np.float64(0.3858500003843801), 'keras': np.float64(1.3195000001360313)}


,diff,lat_ratio,thr_ratio,param_diff
amzn_rnn,0.0803,2.6035,0.6959,-64.0
amzn_lstm,0.0012,4.6245,1.8818,-256.0
amzn_gru,-0.0088,1.7680,1.1488,0.0
amzn_bilstm,0.0113,6.2904,2.1195,-512.0
kkbox_rnn,-0.0016,2.4523,0.7321,-64.0
kkbox_lstm,-0.0049,4.3230,1.8787,-256.0
kkbox_gru,0.0035,1.7501,0.9378,0.0
kkbox_bilstm,-0.0008,5.2568,2.3257,-512.0


## Dữ liệu hình và metadata cho Web App

In [7]:
for ds, metric in (("amzn", "rmse"), ("kkbox", "f1")):
    for fw in FW:
        write_dat(f"bench_{ds}_{fw}", {"lat": [res[f"{ds}_{fw}_{c}"]["latency_ms"] for c in CELLS],
                                       "m": [res[f"{ds}_{fw}_{c}"][metric] * (100 if ds == "kkbox" else 1) for c in CELLS],
                                       "label": [CELL_LABELS[c].replace(" ", "~") for c in CELLS]})
write_dat("bench_size", {"i": list(range(4)),
                         **{f"{fw}_params": [res[f"amzn_{fw}_{c}"]["params"] / 1000 for c in CELLS] for fw in FW},
                         **{f"{fw}_kb": [res[f"amzn_{fw}_{c}"]["size_kb"] for c in CELLS] for fw in FW}})

# Hai người dùng thật trong tập test cho Web App: churn có xác suất cao nhất, loyal có xác suất churn thấp nhất
bt = best["kkbox_torch"]
prob = sigmoid(load_preds(f"kkbox_torch_{bt}")[1])
yt = C["y"]["test"]
i_churn = int(np.flatnonzero(yt == 1)[np.argmax(prob[yt == 1])])
i_loyal = int(np.flatnonzero(yt == 0)[np.argmin(prob[yt == 0])])
raw_test = C["raw"][C["idx"]["test"]]
presets = {"churn": {"label": "Người dùng thật đã churn", "sequence": raw_test[i_churn].round(5), "is_churn": 1,
                     "prob_best": prob[i_churn]},
           "loyal": {"label": "Người dùng thật ở lại", "sequence": raw_test[i_loyal].round(5), "is_churn": 0,
                     "prob_best": prob[i_loyal]}}
thresholds = {f"kkbox_{fw}_{c}": res[f"kkbox_{fw}_{c}"]["threshold"] for fw in FW for c in CELLS}
merge_metadata({"best": best, "thresholds": thresholds, "presets": presets,
                "scalers": {"amzn": S["scaler"].state(), "kkbox": C["scaler"].state()},
                "seq_len": {"amzn": S["X"]["test"].shape[1], "kkbox": C["X"]["test"].shape[1]},
                "kkbox_features": KK_FEATURES, "config": {"train": CFG, "optimizer": OPT}})
print("đã ghi models/metadata.json · preset churn p =", round(float(prob[i_churn]), 4),
      "· preset loyal p =", round(float(prob[i_loyal]), 4))

đã ghi models/metadata.json · preset churn p = 0.9805 · preset loyal p = 0.2348


In [8]:
save_metrics("bench", {"models": res, "naive": naive, "cmp": cmp, "lat_median": lat, "best": best,
                       "best_test": best_test, "n_beats_naive": len(beats_naive), "beats_naive": beats_naive,
                       "threads": 1, "warmup": 20, "repeat": 200, "batch_big": 256,
                       "torch_version": torch.__version__, "keras_version": keras.__version__})
print("đã ghi outputs/metrics/bench.json")

đã ghi outputs/metrics/bench.json
